# Notebook 1 — Data Ingestion & Transformation Pipeline
This notebook ingests TransXChange XML timetables, GTFS-Realtime telemetry, and SIRI-SX disruptions at the **Service / Route level** to preserve scale ($N > 10,000$) and build a consolidated master dataset.

In [2]:
# SECTION 1 — Environment Setup & Spark Initialization
import os
import sys
import glob
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
import pandas as pd

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("Bus Service Benchmarking - ETL Pipeline")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.driver.memory", "6g")
    .config("spark.executor.memory", "6g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark Session Created Successfully. Version:", spark.version)

Spark Session Created Successfully. Version: 4.1.1


In [2]:
# SECTION 2 — Parse TransXChange XML Timetable Dataset (Service Level)
BASE_PATH = r"D:\big data assgn\bodds_archive_20260727"
xml_files = list(Path(BASE_PATH).rglob("*.xml"))
print(f"Total Timetable XML Files Discovered: {len(xml_files)}")

NAMESPACE = {"txc": "http://www.transxchange.org.uk/"}
records = []

for xml_file in tqdm(xml_files, desc="Parsing Timetable XMLs"):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        operator_folder = Path(xml_file).parent.name
        operator_name = operator_folder.rsplit("_", 1)[0]

        service_code = None
        line_name = None
        route_description = None

        service = root.find(".//txc:Service", NAMESPACE)
        if service is not None:
            service_code_elem = service.find("txc:ServiceCode", NAMESPACE)
            if service_code_elem is not None:
                service_code = service_code_elem.text

            line_elem = service.find(".//txc:LineName", NAMESPACE)
            if line_elem is not None:
                line_name = line_elem.text

            desc_elem = service.find(".//txc:Description", NAMESPACE)
            if desc_elem is not None:
                route_description = desc_elem.text

        vehicle_journeys = root.findall(".//txc:VehicleJourney", NAMESPACE)
        trip_count = len(vehicle_journeys)

        stop_points = root.findall(".//txc:AnnotatedStopPointRef", NAMESPACE)
        stop_count = len(stop_points)

        records.append({
            "Operator": operator_name,
            "ServiceCode": service_code if service_code else "Unknown",
            "LineName": line_name if line_name else "Unknown",
            "Description": route_description if route_description else "Unknown",
            "Trips": trip_count,
            "Stops": stop_count,
            "XML_File": str(xml_file)
        })
    except Exception:
        continue

# Convert extracted records to PySpark DataFrame
records_df = pd.DataFrame(records)
services_spark = spark.createDataFrame(records_df)

# Type casting & cleaning duplicates
services_spark = services_spark \
    .withColumn("Trips", col("Trips").cast("int")) \
    .withColumn("Stops", col("Stops").cast("int")) \
    .dropDuplicates()

# Fill missing values
services_spark = services_spark.fillna({
    "ServiceCode": "Unknown",
    "LineName": "Unknown",
    "Description": "Unknown"
})

print(f"Total Unique Services Loaded: {services_spark.count()}")

Total Timetable XML Files Discovered: 19306


Parsing Timetable XMLs: 100%|██████████| 19306/19306 [1:00:03<00:00,  5.36it/s]


Total Unique Services Loaded: 19304


In [3]:
# SECTION 3 — Ingest & Aggregate GTFS-Realtime Data by Route
from google.transit import gtfs_realtime_pb2

GTFS_PATH = r"D:\big data assgn\gtfsrt_2026-07-28_094937\gtfsrt.bin"
feed = gtfs_realtime_pb2.FeedMessage()
with open(GTFS_PATH, "rb") as f:
    feed.ParseFromString(f.read())

vehicle_records = []
for entity in feed.entity:
    if entity.HasField("vehicle"):
        v = entity.vehicle
        vehicle_records.append({
            "VehicleID": v.vehicle.id,
            "TripID": v.trip.trip_id,
            "RouteID": v.trip.route_id,
            "Latitude": v.position.latitude,
            "Longitude": v.position.longitude,
            "Bearing": v.position.bearing,
            "Speed": v.position.speed,
            "Timestamp": v.timestamp
        })

vehicle_pdf = pd.DataFrame(vehicle_records)
vehicle_spark = spark.createDataFrame(vehicle_pdf).dropDuplicates()

# Aggregate real-time telemetry metrics BY ROUTE (RouteID / LineName)
route_telemetry = vehicle_spark.groupBy("RouteID").agg(
    count("*").alias("RouteVehicleCount"),
    avg("Speed").alias("RouteAvgSpeed"),
    max("Speed").alias("RouteMaxSpeed")
)

print(f"Total Routes with GTFS Telemetry: {route_telemetry.count()}")

Total Routes with GTFS Telemetry: 5312


In [4]:
# SECTION 4 — Ingest & Parse SIRI-SX Disruption Data
DISRUPTION_FILE = r"D:\big data assgn\sirisx_2026-07-28_011019\sirisx.xml"
dis_tree = ET.parse(DISRUPTION_FILE)
dis_root = dis_tree.getroot()

dis_records = []
for element in dis_root.iter():
    tag = element.tag.split("}")[-1]
    if tag == "PtSituationElement":
        rec = {}
        for child in element:
            child_tag = child.tag.split("}")[-1]
            rec[child_tag] = child.text
        dis_records.append(rec)

dis_pdf = pd.DataFrame(dis_records)
disruption_spark = spark.createDataFrame(dis_pdf).dropDuplicates()

if "LineRef" in disruption_spark.columns:
    route_disruptions = disruption_spark.groupBy("LineRef").agg(
        count("*").alias("RouteDisruptions"),
        sum(when(col("Planned") == "false", 1).default(0)).alias("UnplannedDisruptions")
    )
else:
    dis_summary = disruption_spark.agg(
        count("*").alias("RouteDisruptions")
    ).first().asDict()
    route_disruptions = None

In [5]:
# SECTION 5 — Join Datasets into Master Service-Level Dataset
master_benchmark = services_spark.join(
    route_telemetry,
    services_spark["LineName"] == route_telemetry["RouteID"],
    how="left"
)

if route_disruptions is not None:
    master_benchmark = master_benchmark.join(
        route_disruptions,
        master_benchmark["LineName"] == route_disruptions["LineRef"],
        how="left"
    )
else:
    master_benchmark = master_benchmark.withColumn(
        "RouteDisruptions", lit(dis_summary["RouteDisruptions"])
    )

master_benchmark = master_benchmark.fillna({
    "RouteVehicleCount": 0,
    "RouteAvgSpeed": 0.0,
    "RouteMaxSpeed": 0.0,
    "RouteDisruptions": 0
})

# Feature Engineering: Compute Service Complexity & Performance Indices
master_benchmark = master_benchmark.withColumn(
    "ServicePerformanceIndex",
    round(
        (col("Trips") * 0.4) + 
        (col("Stops") * 0.3) + 
        (col("RouteAvgSpeed") * 0.2) - 
        (col("RouteDisruptions") * 0.1), 
        2
    )
)

print("\n--- MASTER SERVICE BENCHMARK DATASET ---")
print(f"Total Rows (Services): {master_benchmark.count()}")
master_benchmark.printSchema()

os.makedirs("output", exist_ok=True)
master_benchmark.write.mode("overwrite").parquet("output/benchmark_dataset")
print("Dataset successfully stored to output/benchmark_dataset")


--- MASTER SERVICE BENCHMARK DATASET ---
Total Rows (Services): 19304
root
 |-- Operator: string (nullable = true)
 |-- ServiceCode: string (nullable = false)
 |-- LineName: string (nullable = false)
 |-- Description: string (nullable = false)
 |-- Trips: integer (nullable = true)
 |-- Stops: integer (nullable = true)
 |-- XML_File: string (nullable = true)
 |-- RouteID: string (nullable = true)
 |-- RouteVehicleCount: long (nullable = false)
 |-- RouteAvgSpeed: double (nullable = false)
 |-- RouteMaxSpeed: double (nullable = false)
 |-- RouteDisruptions: integer (nullable = false)
 |-- ServicePerformanceIndex: double (nullable = true)

Dataset successfully stored to output/benchmark_dataset
